# RAG Pipeline with Embeddings: Practice Exercise - SOLUTION

This notebook contains the solutions for the RAG pipeline practice exercise.

**What you'll implement:**
- The RAG state class to track query, context, and answer
- The retrieve node for similarity search
- The generate node for creating grounded responses
- The LangGraph workflow connecting all components

## Setup

Run this cell to import all required libraries and initialize the API client.

In [1]:
# Setup - run this cell first

import os
import csv
from dotenv import load_dotenv
from typing import TypedDict, List

# LangChain imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

# LangGraph imports
from langgraph.graph import StateGraph, START, END

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please set it in your .env file.")

print("Setup complete!")

Setup complete!


## Context

You are building a technical support assistant for TechMart that can answer customer troubleshooting questions. The knowledge base contains troubleshooting guides for various tech products including laptops, monitors, mice, and desktop computers.

**Your task:**
1. Define the RAG state class to track the workflow data
2. Initialize the LLM and create an appropriate prompt template
3. Implement the retrieve node that finds relevant troubleshooting guides
4. Implement the generate node that creates helpful answers
5. Build the LangGraph workflow connecting these components

**Input:** A customer question about a technical issue (string)

**Output:** A grounded troubleshooting response based on the knowledge base

## Knowledge Base Setup (Provided)

The following cells load the TechMart troubleshooting data and create the vector store. Run them to prepare the knowledge base.

In [2]:
# Load TechMart troubleshooting data - run this cell

data_path = "../../data/techmart_troubleshooting.csv"

troubleshooting_guides = []
with open(data_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        troubleshooting_guides.append(row)

print(f"Loaded {len(troubleshooting_guides)} troubleshooting guides")
print(f"\nColumns: {list(troubleshooting_guides[0].keys())}")
print(f"\nSample guide:")
sample = troubleshooting_guides[0]
print(f"  Issue: {sample['issue_title']}")
print(f"  Product: {sample['product_id']}")
print(f"  Symptoms: {sample['symptoms'][:100]}...")

Loaded 154 troubleshooting guides

Columns: ['issue_title', 'product_id', 'symptoms', 'steps']

Sample guide:
  Issue: Wireless mouse disconnects randomly
  Product: SKU010
  Symptoms: Cursor freezes for 1–2 seconds, USB‑dongle LED blinks red, occurs every 5–10 minutes....


In [3]:
# Create documents and build vector store - run this cell

# Convert to LangChain Documents
# Each document combines issue, symptoms, and troubleshooting steps
support_docs = []
for guide in troubleshooting_guides:
    content = f"""Issue: {guide['issue_title']}
Symptoms: {guide['symptoms']}
Troubleshooting Steps:
{guide['steps']}"""
    
    doc = Document(
        page_content=content,
        metadata={
            "issue_title": guide["issue_title"],
            "product_id": guide["product_id"]
        }
    )
    support_docs.append(doc)

# Split into chunks for better retrieval
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)
support_chunks = splitter.split_documents(support_docs)

# Create embeddings and vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

support_vectorstore = Chroma(
    collection_name="techmart_support",
    embedding_function=embeddings
)
support_vectorstore.add_documents(documents=support_chunks)

print(f"Created {len(support_chunks)} chunks from {len(support_docs)} guides")
print("Vector store ready!")

Created 305 chunks from 154 guides
Vector store ready!


In [4]:
# Test the vector store - run this cell to verify it works

test_query = "My laptop battery drains too fast"
test_results = support_vectorstore.similarity_search(test_query, k=2)

print(f"Test query: '{test_query}'")
print(f"\nFound {len(test_results)} results:")
for i, doc in enumerate(test_results, 1):
    print(f"\n[{i}] {doc.metadata.get('issue_title', 'Unknown')}")
    print(f"    {doc.page_content[:150]}...")

Test query: 'My laptop battery drains too fast'

Found 2 results:

[1] UltraBook Pro battery drains quickly
    Issue: UltraBook Pro battery drains quickly
Symptoms: Battery indicator drops from 100 % to 70 % in under an hour of web browsing.
Troubleshooting Ste...

[2] Laptop Battery Not Charging When Plugged In
    4. Disconnect the charger, power off the laptop, and hold the power button for 15 seconds. Then reconnect the charger and power on the device. 
5. In ...


## Part 1: Define the RAG State - SOLUTION

Define the state class that will be passed between nodes in your workflow.

In [5]:
class SupportRAGState(TypedDict):
    """
    State for the TechMart support RAG workflow.
    
    This TypedDict contains three fields:
    - query: The customer's support question (str)
    - context: Retrieved troubleshooting documents (List[Document])
    - answer: The generated support response (str)
    """
    # SOLUTION: Define the three state fields with appropriate types
    query: str
    context: List[Document]
    answer: str

print("SupportRAGState defined!")

SupportRAGState defined!


### Understanding the Solution

The `SupportRAGState` TypedDict defines the schema for data flowing through the RAG workflow:

1. **`query: str`** - The customer's original question. This is the input to the workflow and is used by the retrieve node to find relevant documents.

2. **`context: List[Document]`** - The retrieved documents from the vector store. Using `List[Document]` (from langchain_core.documents) preserves both the content and metadata of each document.

3. **`answer: str`** - The final generated response. This is populated by the generate node and represents the output of the workflow.

Using TypedDict provides type hints that help catch errors and makes the code more maintainable.

## Part 2: Initialize LLM and Prompt - SOLUTION

Create the language model instance and define a prompt template for technical support responses.

In [6]:
# SOLUTION 1: Initialize the ChatOpenAI model
# Using gpt-4o-mini for cost efficiency and temperature=0 for consistent responses
support_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# SOLUTION 2: Create a ChatPromptTemplate for technical support
# The template instructs the LLM to act as a tech support agent
# and only use the provided context to answer questions
support_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful TechMart technical support assistant. 
Your job is to help customers troubleshoot their tech products.

Use ONLY the following troubleshooting guides to answer the customer's question.
If the guides don't contain relevant information, say you don't have specific 
troubleshooting steps for that issue and recommend contacting support.

Be friendly, clear, and provide step-by-step instructions when possible.

Troubleshooting Guides:
{context}"""),
    ("human", "{query}")
])

print("LLM and prompt initialized!")

LLM and prompt initialized!


### Understanding the Solution

The LLM and prompt setup has two key components:

1. **ChatOpenAI initialization:**
   - `model="gpt-4o-mini"`: A capable but cost-efficient model for support tasks
   - `temperature=0`: Ensures consistent, deterministic responses (important for support)

2. **ChatPromptTemplate:**
   - **System message**: Sets the persona (TechMart support assistant) and constraints (only use provided context)
   - **`{context}` placeholder**: Where retrieved documents will be inserted
   - **Human message with `{query}`**: The customer's actual question
   - The instruction to say "don't have specific troubleshooting steps" prevents hallucination when context is insufficient

## Part 3: Implement the Nodes - SOLUTION

Implement the retrieve and generate nodes for the RAG workflow.

### Retrieve Node

This node takes the customer query and retrieves relevant troubleshooting documents.

In [7]:
def retrieve_support_docs(state: SupportRAGState) -> SupportRAGState:
    """
    Retrieve relevant troubleshooting documents based on customer query.
    
    This function:
    1. Extracts the query from the state
    2. Uses support_vectorstore.similarity_search() to find 4 relevant documents
    3. Returns a new state dict with query, context (retrieved docs), and empty answer
    
    Args:
        state: Current workflow state containing the customer query
    
    Returns:
        Updated state dict with 'context' populated with retrieved Document objects
    """
    # SOLUTION: Implement retrieval logic
    
    # 1. Get query from state
    query = state["query"]
    
    # 2. Call support_vectorstore.similarity_search(query, k=4)
    retrieved_docs = support_vectorstore.similarity_search(query, k=4)
    
    print(f"Retrieved {len(retrieved_docs)} documents for query: '{query[:50]}...'")
    
    # 3. Return dict with query, context, and empty answer
    return {
        "query": query,
        "context": retrieved_docs,
        "answer": ""
    }

print("retrieve_support_docs node defined!")

retrieve_support_docs node defined!


### Generate Node

This node uses the retrieved context to generate a helpful support response.

In [8]:
def generate_support_response(state: SupportRAGState) -> SupportRAGState:
    """
    Generate a support response using retrieved troubleshooting context.
    
    This function:
    1. Extracts query and context documents from state
    2. Formats context docs into a single string (includes issue title and content)
    3. Uses support_prompt.invoke() with context and query
    4. Calls support_llm.invoke() with the formatted prompt
    5. Returns updated state with the answer
    
    Args:
        state: Current workflow state with query and retrieved context
    
    Returns:
        Updated state dict with 'answer' populated
    """
    # SOLUTION: Implement generation logic
    
    # 1. Get query and context from state
    query = state["query"]
    context_docs = state["context"]
    
    # 2. Format context: join docs with titles
    context_parts = []
    for doc in context_docs:
        title = doc.metadata.get('issue_title', 'Unknown Issue')
        context_parts.append(f"[Issue: {title}]\n{doc.page_content}")
    
    context_str = "\n\n---\n\n".join(context_parts)
    
    # 3. Invoke prompt with context and query
    prompt_value = support_prompt.invoke({
        "context": context_str,
        "query": query
    })
    
    # 4. Invoke LLM with the prompt result
    response = support_llm.invoke(prompt_value)
    
    print("Generated support response")
    
    # 5. Return dict with query, context, and answer
    return {
        "query": query,
        "context": context_docs,
        "answer": response.content
    }

print("generate_support_response node defined!")

generate_support_response node defined!


### Understanding the Node Solutions

**Retrieve Node:**
- Extracts the query from the incoming state
- Uses `similarity_search(query, k=4)` to find the 4 most relevant documents
- Returns a state dict that updates the `context` field with retrieved documents

**Generate Node:**
- Formats retrieved documents into a readable string with issue titles as headers
- Uses `---` separators between documents for clarity
- Invokes the prompt template with context and query placeholders filled in
- Passes the formatted prompt to the LLM and extracts `response.content`

Both nodes follow the LangGraph pattern of:
1. Reading from the input state
2. Performing their specific operation
3. Returning a dict with updated state fields

## Part 4: Build the Workflow - SOLUTION

Create the LangGraph workflow that connects the retrieve and generate nodes.

In [9]:
# Build the RAG workflow

# SOLUTION 1: Create a StateGraph with SupportRAGState
support_workflow = StateGraph(SupportRAGState)

# SOLUTION 2: Add the retrieve node
support_workflow.add_node("retrieve", retrieve_support_docs)

# SOLUTION 3: Add the generate node
support_workflow.add_node("generate", generate_support_response)

# SOLUTION 4: Add edge from START to "retrieve"
support_workflow.add_edge(START, "retrieve")

# SOLUTION 5: Add edge from "retrieve" to "generate"
support_workflow.add_edge("retrieve", "generate")

# SOLUTION 6: Add edge from "generate" to END
support_workflow.add_edge("generate", END)

# Compile the workflow
support_rag = support_workflow.compile()

print("Workflow compiled!")
print("\nWorkflow structure: START -> retrieve -> generate -> END")

Workflow compiled!

Workflow structure: START -> retrieve -> generate -> END


### Understanding the Workflow Solution

The workflow is built using LangGraph's `StateGraph`:

1. **`StateGraph(SupportRAGState)`** - Creates a graph that will pass `SupportRAGState` between nodes

2. **`add_node(name, function)`** - Registers each node function with a name:
   - `"retrieve"` -> `retrieve_support_docs`
   - `"generate"` -> `generate_support_response`

3. **`add_edge(from, to)`** - Defines the flow:
   - `START -> "retrieve"`: Begin with retrieval
   - `"retrieve" -> "generate"`: Pass retrieved docs to generation
   - `"generate" -> END`: Complete after generating response

4. **`compile()`** - Converts the graph definition into an executable application

This creates a simple linear pipeline: query comes in, documents are retrieved, response is generated.

## Run Your Implementation

Test your RAG pipeline with sample customer questions.

In [10]:
# Helper function to query the support system

def ask_support(question: str):
    """Query the TechMart support RAG system."""
    print("=" * 70)
    print(f"Customer Question: {question}")
    print("=" * 70)
    
    result = support_rag.invoke({
        "query": question,
        "context": [],
        "answer": ""
    })
    
    print("\nSources consulted:")
    for doc in result["context"]:
        print(f"  - {doc.metadata.get('issue_title', 'Unknown')} ({doc.metadata.get('product_id', 'Unknown')})")
    
    print(f"\nSupport Response:\n{result['answer']}")
    print("\n")

In [11]:
# Test 1: Battery issue
ask_support("My UltraBook Pro battery is draining really fast, what should I do?")

Customer Question: My UltraBook Pro battery is draining really fast, what should I do?
Retrieved 4 documents for query: 'My UltraBook Pro battery is draining really fast, ...'
Generated support response

Sources consulted:
  - UltraBook Pro battery drains quickly (SKU001)
  - UltraBook Pro overheating under load (SKU001)
  - UltraBook Air fan noise while idle (SKU002)
  - UltraBook Air Wi‑Fi disconnects frequently (SKU002)

Support Response:
I'm sorry to hear that your UltraBook Pro battery is draining quickly. Here are some troubleshooting steps you can follow to help improve the battery life:

1. **Switch Windows power mode to ‘Balanced’**:
   - Go to the Control Panel.
   - Click on "Power Options."
   - Select the "Balanced" power plan.

2. **Disable the OLED panel’s HDR in Display Settings**:
   - Right-click on the desktop and select "Display settings."
   - Scroll down to "Windows HD Color" and turn off HDR.

3. **Check Task Manager for apps with >10% CPU usage and close them**:

In [12]:
# Test 2: Connectivity issue
ask_support("My wireless mouse keeps disconnecting every few minutes")

Customer Question: My wireless mouse keeps disconnecting every few minutes
Retrieved 4 documents for query: 'My wireless mouse keeps disconnecting every few mi...'
Generated support response

Sources consulted:
  - Wireless mouse disconnects randomly (SKU010)
  - Mouse Pointer Lagging or Stuttering During Use (SKU236)
  - Mouse Pointer Lag or Stuttering in Wireless Mode (SKU035)
  - UltraBook Air Wi‑Fi disconnects frequently (SKU002)

Support Response:
I'm sorry to hear that your wireless mouse is disconnecting. Here are some troubleshooting steps you can follow to resolve the issue:

1. **Move the Receiver**: Try moving the USB receiver to a front USB port on your computer to reduce interference.

2. **Check the Battery**: Replace the AA battery or charge the mouse for at least 30 minutes to ensure it has enough power.

3. **Toggle the Switch**: If your mouse has a 2.4 GHz / Bluetooth switch, toggle it and re-pair the mouse with your computer.

4. **Update Firmware**: Use the GlideMou

In [13]:
# Test 3: Display issue
ask_support("The monitor won't turn on after my computer wakes from sleep")

Customer Question: The monitor won't turn on after my computer wakes from sleep
Retrieved 4 documents for query: 'The monitor won't turn on after my computer wakes ...'
Generated support response

Sources consulted:
  - VisionView 27 monitor won’t turn on (SKU005)
  - No Signal Detected on HDMI 2.1 Port (SKU205)
  - No Signal Detected on DisplayPort Connection (SKU240)
  - Desktop Powers On But No Display Output (SKU068)

Support Response:
It sounds like you're experiencing an issue with your monitor not turning on after your computer wakes from sleep. Here are some troubleshooting steps you can follow:

1. **Verify the power brick**: Make sure the power brick is firmly clicked into the monitor port.

2. **Test a different outlet**: Try plugging the monitor into a different power outlet to see if that resolves the issue.

3. **Hard-reset the monitor**: Hold the monitor power button for 10 seconds to perform a hard reset.

4. **Swap the power cable**: If the power LED stays off after sw

## Summary

In this exercise, you built a complete RAG pipeline for TechMart technical support:

1. **SupportRAGState** - A TypedDict defining the workflow state with `query`, `context`, and `answer` fields

2. **LLM and Prompt** - ChatOpenAI with a system prompt that instructs the model to act as a support agent and only use provided context

3. **Retrieve Node** - Uses vector similarity search to find relevant troubleshooting guides

4. **Generate Node** - Formats context and invokes the LLM to produce grounded responses

5. **LangGraph Workflow** - Connects nodes in a linear pipeline: START -> retrieve -> generate -> END

This pattern can be extended with:
- Additional nodes for query rewriting or response validation
- Conditional edges for routing different types of questions
- Human-in-the-loop interrupts for complex issues